In [ ]:
!pip install -U transformers accelerate peft bitsandbytes datasets tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
  

In [ ]:
import os
import json
import math
import random
import re
import shutil
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    pipeline
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from google.colab import files

In [ ]:
ALPACA_TMPL = (
    "### Instruction:\n"
    "Answer as an endodontist using only the given context. "
    "Preserve exact clinical and endodontic keywords from the context whenever possible. "
    "Do not replace technical terms with general language. "
    "Use abbreviations such as NaOCl, EDTA, MTA exactly if they appear in the context.\n\n"
    "### Input:\n"
    "CONTEXT:\n{context}\n\n"
    "Question: {question}\n\n"
    "### Response:\n"
)

def format_for_training(context: str, question: str, answer: str) -> str:
    return ALPACA_TMPL.format(context=context, question=question) + answer

def format_for_inference(context: str, question: str) -> str:
    return ALPACA_TMPL.format(context=context, question=question)

In [ ]:
qa_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_id,
    trust_remote_code=True
)

if qa_tokenizer.pad_token is None:
    qa_tokenizer.pad_token = qa_tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

qa_model = AutoModelForCausalLM.from_pretrained(
    qa_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

qa_model.config.use_cache = False

print("Qwen QA generation modeli yüklendi.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen QA generation modeli yüklendi.


In [ ]:
chunks_path = Path("/content/chunks (5).jsonl")
qa_output_path = Path("/content/raw_qa_dataset_qwen.jsonl")

rows = []
with chunks_path.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if "text" in row and len(row["text"].strip()) > 250:
            rows.append(row)

print(f"Uygun chunk sayısı: {len(rows)}")
print(rows[0]["text"][:500])

Uygun chunk sayısı: 6584
Cohen’s Pathways of the Pulp TWELFTH EDITION EDITORS Louis H. Berman, DDS, FACD Clinical Associate Professor, Department of Endodontics, School of Dentistry, University of Maryland, Baltimore, Maryland Faculty, Albert Einstein Medical Center, Philadelphia, Pennsylvania Private Practice, Annapolis Endodontics, Annapolis, Maryland Diplomate, American Board of Endodontics Kenneth M. Hargreaves, DDS, PhD, FICD, FACD Professor and Chair, Department of Endodontics Professor, Departments of Pharmacolog


In [ ]:
qa_generator = pipeline(
    "text-generation",
    model=qa_model,
    tokenizer=qa_tokenizer
)

QA_GENERATION_PROMPT = """You are a world-class Endodontist.
Read the given text and produce exactly one question-answer pair.

STRICT FORMAT:
Question: ...
Answer: ...

Rules:
- The question must be answerable only from the given text.
- The answer must be short, clear, and highly technical.
- The answer must contain at least 2 important endodontic or clinical keywords copied from the text.
- Do not replace technical terms with simpler synonyms.
- If the text contains abbreviations such as EDTA, NaOCl, MTA, use them exactly.
- If you do not follow the format exactly, your answer is invalid.

Text:
{context}
"""

In [ ]:
def generate_qa(context: str):
    prompt = QA_GENERATION_PROMPT.format(context=context[:1000])

    outputs = qa_generator(
        prompt,
        max_new_tokens=140,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )

    raw = outputs[0]["generated_text"].strip()
    raw = re.sub(r"\r\n", "\n", raw).strip()

    q_match = re.search(r"Question\s*:\s*(.+?)(?:\n|$)", raw, re.IGNORECASE | re.DOTALL)
    a_match = re.search(r"Answer\s*:\s*(.+)", raw, re.IGNORECASE | re.DOTALL)

    if not q_match or not a_match:
        lines = [line.strip() for line in raw.split("\n") if line.strip()]

        if len(lines) >= 2:
            question = lines[0]
            answer = " ".join(lines[1:])

            question = re.sub(r"^question\s*:?\s*", "", question, flags=re.IGNORECASE).strip()
            answer = re.sub(r"^answer\s*:?\s*", "", answer, flags=re.IGNORECASE).strip()
        else:
            return None
    else:
        question = q_match.group(1).strip()
        answer = a_match.group(1).strip()

    question = re.sub(r"^question\s*:?\s*", "", question, flags=re.IGNORECASE).strip()
    answer = re.sub(r"^answer\s*:?\s*", "", answer, flags=re.IGNORECASE).strip()

    if len(question) < 6 or len(answer) < 10:
        return None

    if len(question) > 300 or len(answer) > 1200:
        return None

    return {
        "context": context[:1000],
        "question": question,
        "answer": answer
    }


def save_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [ ]:
import torch
print("CUDA var mı?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA var mı?: True
GPU: Tesla T4


In [ ]:
sample_size = min(2000, len(rows))
sampled_rows = random.sample(rows, sample_size)

raw_qa = []
for i, ch in enumerate(tqdm(sampled_rows, desc="QA generating with Qwen"), 1):
    item = generate_qa(ch["text"])
    if item:
        raw_qa.append(item)

    if i % 10 == 0:
        print(f"{i} işlendi | QA sayısı: {len(raw_qa)}")

print(f"Üretilen örnek sayısı: {len(raw_qa)}")
save_jsonl(qa_output_path, raw_qa)
print(f"Kaydedildi: {qa_output_path}")

QA generating with Qwen:   0%|          | 0/2000 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
QA generating with Qwen:   0%|          | 10/2000 [03:00<5:25:25,  9.81s/it][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the 

10 işlendi | QA sayısı: 6


QA generating with Qwen:   1%|          | 20/2000 [04:21<5:43:54, 10.42s/it][transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


20 işlendi | QA sayısı: 12


QA generating with Qwen:   2%|▏         | 30/2000 [06:23<6:44:53, 12.33s/it][transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

1980 işlendi | QA sayısı: 1169


QA generating with Qwen: 100%|█████████▉| 1990/2000 [5:15:23<01:32,  9.24s/it][transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1990 işlendi | QA sayısı: 1178


QA generating with Qwen: 100%|██████████| 2000/2000 [5:17:08<00:00,  9.51s/it]

2000 işlendi | QA sayısı: 1183
Üretilen örnek sayısı: 1183
Kaydedildi: /content/raw_qa_dataset_qwen.jsonl


In [ ]:
for i in range(min(5, len(raw_qa))):
    print("\n" + "=" * 80)
    print("QUESTION:", raw_qa[i]["question"])
    print("ANSWER:", raw_qa[i]["answer"])


QUESTION: What is the main concern regarding the use of dental sealants on young permanent teeth?
ANSWER: The main concern is that using dental sealants in young permanent teeth increases the risk of caries among children. This conclusion is drawn based on the study's findings that direct contact with dentin adhesives or composite resins weakens the pulp's defense against invading microorganisms, leading

QUESTION: What factor did the London Eastman study find to independently and significantly affect periapical healing? Answer: Canals being filled to the same extent as canal preparation.
ANSWER: Canals being filled to the same extent as canal preparation.

QUESTION: What does the forest plot show about the probability of periapical health for teeth undergoing root canal treatment?
ANSWER: The forest plot displays the combined and individual studies' probabilities of periapical health for teeth undergoing root canal treatment across different decades and criteria for success. It provi

In [ ]:
raw_qa = []
with open(qa_output_path, "r", encoding="utf-8") as f:
    for line in f:
        raw_qa.append(json.loads(line))

print("Yüklenen QA sayısı:", len(raw_qa))

Yüklenen QA sayısı: 1183


In [ ]:
train_data = Dataset.from_list(raw_qa)
print(train_data)

Dataset({
    features: ['context', 'question', 'answer'],
    num_rows: 1183
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

ft_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

ft_tokenizer = AutoTokenizer.from_pretrained(ft_model_id)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

bnb_config_ft = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

ft_model = AutoModelForCausalLM.from_pretrained(
    ft_model_id,
    quantization_config=bnb_config_ft,
    device_map="auto"
)

ft_model.config.use_cache = False
print("Fine-tuning için TinyLlama yüklendi.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Fine-tuning için TinyLlama yüklendi.


In [ ]:
raw_qa = []
with open(qa_output_path, "r", encoding="utf-8") as f:
    for line in f:
        raw_qa.append(json.loads(line))

print("Yüklenen QA sayısı:", len(raw_qa))

train_data = Dataset.from_list(raw_qa)
print(train_data)

Yüklenen QA sayısı: 1183
Dataset({
    features: ['context', 'question', 'answer'],
    num_rows: 1183
})


In [ ]:
ALPACA_TMPL = (
    "### Instruction:\n"
    "Answer as an endodontist using only the given context. "
    "Preserve exact clinical and endodontic keywords from the context whenever possible. "
    "Do not replace technical terms with general language. "
    "Use abbreviations such as NaOCl, EDTA, MTA exactly if they appear in the context.\n\n"
    "### Input:\n"
    "CONTEXT:\n{context}\n\n"
    "Question: {question}\n\n"
    "### Response:\n"
)

In [ ]:
MAX_LENGTH = 768

def tokenize_with_response_only_loss(example):
    prompt = ALPACA_TMPL.format(
        context=example["context"],
        question=example["question"]
    )
    full_text = prompt + example["answer"]

    full_tokens = ft_tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    prompt_tokens = ft_tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = input_ids.copy()
    prompt_len = len(prompt_tokens["input_ids"])

    labels[:prompt_len] = [-100] * prompt_len

    labels = [
        lbl if attn == 1 else -100
        for lbl, attn in zip(labels, attention_mask)
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_data = train_data.map(
    tokenize_with_response_only_loss,
    remove_columns=train_data.column_names
)

print("Tokenization tamamlandı.")

Map:   0%|          | 0/1183 [00:00<?, ? examples/s]

Tokenization tamamlandı.


In [ ]:
ft_model = prepare_model_for_kbit_training(ft_model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()

trainable params: 9,011,200 || all params: 1,109,059,584 || trainable%: 0.8125


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./small_model_fine_tuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1.5e-4,
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=ft_tokenizer,
        model=ft_model,
        padding=True
    )
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

trainer.model.save_pretrained("./small_model_fine_tuned")
ft_tokenizer.save_pretrained("./small_model_fine_tuned")

print("Fine-tuning tamamlandı.")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.231531
20,1.038516
30,1.035567
40,1.081152
50,0.891219
60,0.908155
70,0.863471
80,0.862761
90,0.859166
100,0.890536


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Fine-tuning tamamlandı.


In [ ]:
import os
import zipfile

model_dir = "./small_model_fine_tuned"
zip_path = "./small_model_fine_tuned.zip"

def zip_model_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, folder_path)
                zipf.write(full_path, arcname)

zip_model_folder(model_dir, zip_path)

print("✅ Model zip oluşturuldu:", zip_path)

✅ Model zip oluşturuldu: ./small_model_fine_tuned.zip


In [ ]:
TEST_SORULARI = [
    "What is the main purpose of EDTA in endodontics?",
    "What are the primary objectives of using Sodium Hypochlorite (NaOCl)?",
    "What are the clinical signs of irreversible pulpitis?",
    "How should a clinician manage a separated instrument in the root canal?",
    "Explain the importance of working length determination in endodontics."
]

REFERANSLAR = [
    "EDTA is a chelating agent used to remove the inorganic smear layer and soften dentin walls, improving penetration of irrigants.",
    "NaOCl dissolves organic tissue and has broad-spectrum antimicrobial activity, making it the gold standard irrigant.",
    "Irreversible pulpitis presents as sharp lingering pain to thermal stimuli and may include spontaneous pain.",
    "Management of a separated instrument includes bypass attempts, ultrasonic retrieval, or cleaning to the file level.",
    "Working length determination ensures instrumentation stays within the root canal system, preventing apical damage."
]

In [ ]:
print("Cevaplar üretiliyor...")
tahminler = []

def asistan_test_et(soru, test_context):
    prompt = format_for_inference(test_context, soru)
    output = qa_generator(
        prompt,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )
    return output[0]["generated_text"].strip()

test_context = rows[0]["text"][:1200]

for soru in TEST_SORULARI:
    cevap = asistan_test_et(soru, test_context)
    tahminler.append(cevap)
    print(f"\nSORU: {soru}")
    print(f"CEVAP: {cevap}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Cevaplar üretiliyor...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SORU: What is the main purpose of EDTA in endodontics?
CEVAP: EDTA serves a crucial role in endodontic treatment by facilitating debridement and promoting the removal of residual debris from the root canal system during cleaning procedures. Its primary function involves creating a buffer solution that helps to dissolve organic materials like calculus or smear layer, making it easier for instruments to remove these substances without causing damage to the dental pulp tissue. Additionally, EDTA's ability to neutralize chlorhexidine solutions enhances its effectiveness in achieving a cleaner environment within the root canals, thus reducing the


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SORU: What are the primary objectives of using Sodium Hypochlorite (NaOCl)?
CEVAP: The primary objective of using Sodium Hypochlorite (NaOCl) is to provide a highly effective antiseptic solution for root canal therapy, which includes cleaning the root canals and killing bacteria that might be present. This is particularly important due to its high concentration of chlorine ions, which act as disinfectants when diluted into specific concentrations used in these procedures. The use of NaOCl also facilitates better visualization during the procedure by enhancing contrast between tissues, aiding in more precise cleaning and obt


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SORU: What are the clinical signs of irreversible pulpitis?
CEVAP: The clinical signs of irreversible pulpitis include a history of spontaneous or prolonged sharp pain, especially during cold stimulation or temperature changes, along with the following:

1. **Pain** - The most common symptom is severe, often described as a "stabbing" or "sharp" pain that can be triggered by certain stimuli like hot or cold foods or drinks.
2. **Prolonged Pain** - This refers to a lack of relief after treatment has been applied, indicating that the pain persists


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SORU: How should a clinician manage a separated instrument in the root canal?
CEVAP: When faced with the scenario where a separated instrument remains within the root canal during a procedure, it is crucial for the clinician to respond promptly and effectively to prevent further complications. The management plan involves several key steps:

1. **Safety First**: Prioritize patient safety by ensuring that the situation is contained and does not pose immediate risk to the patient. This includes maintaining access to the orofacial structures while minimizing the likelihood of additional trauma or damage to surrounding tissues.

2. **Access Point Selection**:

SORU: Explain the importance of working length determination in endodontics.
CEVAP: The importance of working length determination in endodontics cannot be overstated. It is a fundamental concept that plays a crucial role in ensuring treatment efficacy, minimizing risk of root canal infection, and achieving optimal results in post-t

In [ ]:
!pip install -q rouge-score nltk sentence-transformers

  Preparing metadata (setup.py) ... done


In [ ]:
import nltk
import re
import numpy as np
import math
import torch

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [ ]:
def temizle(metin):
    metin = str(metin).lower()
    metin = re.sub(r"[^\w\s]", " ", metin)
    return re.sub(r"\s+", " ", metin).strip()

smooth = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

cos_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def hesapla_perplexity(model, tokenizer, metin):
    enc = tokenizer(
        metin,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        loss = model(**enc, labels=enc["input_ids"]).loss

    return math.exp(loss.item())

In [ ]:
REFERANSLAR = [
    "EDTA is a chelating agent used to remove the inorganic smear layer and improve dentin permeability.",
    "NaOCl dissolves organic tissue and provides antimicrobial activity.",
    "Irreversible pulpitis causes lingering thermal pain and spontaneous pain.",
    "Separated instrument can be managed by bypass or ultrasonic retrieval.",
    "Working length ensures proper cleaning within root canal system."
]

In [ ]:
df_final = pd.DataFrame({
    "Soru": TEST_SORULARI,
    "Referans_Cevabi": REFERANSLAR,
    "Asistan_Cevabi": tahminler
})

sonuclar = []

for i, row in df_final.iterrows():
    pred = row["Asistan_Cevabi"]
    ref = REFERANSLAR[i]

    pred_t = temizle(pred)
    ref_t = temizle(ref)

    pred_tokens = pred_t.split()
    ref_tokens = ref_t.split()

    # BLEU
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)

    # ROUGE-L
    rouge_l = rouge.score(ref_t, pred_t)["rougeL"].fmeasure

    # METEOR
    meteor = meteor_score([ref_tokens], pred_tokens)

    # Cosine
    vecs = cos_model.encode([ref, pred], normalize_embeddings=True)
    cosine = float(np.dot(vecs[0], vecs[1]))

    # Perplexity
    ppl = hesapla_perplexity(ft_model, ft_tokenizer, pred)

    sonuclar.append({
        "BLEU": round(bleu, 3),
        "ROUGE-L": round(rouge_l, 3),
        "METEOR": round(meteor, 3),
        "Cosine": round(cosine, 3),
        "Perplexity": round(ppl, 3)
    })

df_metrics = pd.DataFrame(sonuclar)
df_metrics

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


,BLEU,ROUGE-L,METEOR,Cosine,Perplexity
0,0.008,0.119,0.255,0.760,6.527
1,0.003,0.067,0.131,0.563,7.181
2,0.006,0.094,0.237,0.861,8.561
3,0.005,0.105,0.244,0.492,11.138
4,0.007,0.112,0.231,0.798,5.662


In [ ]:
print("=== ORTALAMA METRİKLER ===")
print(df_metrics.mean())

=== ORTALAMA METRİKLER ===
BLEU          0.0058
ROUGE-L       0.0994
METEOR        0.2196
Cosine        0.6948
Perplexity    7.8138
dtype: float64


In [ ]:
klinik_sozluk = {
    0: ["edta", "smear", "layer", "chelating"],
    1: ["naocl", "hypochlorite", "antimicrobial"],
    2: ["irreversible", "pulpitis", "pain"],
    3: ["instrument", "bypass", "retrieval"],
    4: ["working", "length", "apical"]
}

def keyword_score(answer, keywords):
    a = temizle(answer)
    hits = [k for k in keywords if k in a]
    return len(hits) / len(keywords), hits

kw_results = []

for i, row in df_final.iterrows():
    score, hits = keyword_score(row["Asistan_Cevabi"], klinik_sozluk[i])
    kw_results.append(score)

print("Keyword Hit Rate:", sum(kw_results)/len(kw_results))

Keyword Hit Rate: 0.6833333333333333


In [ ]:
df_final.to_csv("rag_outputs.csv", index=False)
df_metrics.to_csv("metrics.csv", index=False)

print("CSV kaydedildi.")

CSV kaydedildi.
